# dismantle — FINAL PUSH (all heavy compute)

Single-stream decode is bandwidth-bound (~90 tps ceiling on M3 Pro for q3b; llama.cpp ~50). The only ways past that are the **multiplier** (spec tokens per weight-read) and the **denominator** (fewer bytes/token via sparsity / low-bit). This notebook trains every cloud piece of both, measured by **tps-predictive** metrics.

- **Track B (default ON):** maximize the spec head (accepted-prefix 1.6 → target 2.5+). This is the proven core heavy-compute.
- **Tracks C/D (default OFF, audit before enabling):** sparsity predictor + Q2 draft — experimental, depend on local capture.

`Runtime → Run all`. Best head per model + leaderboard land in Drive.

## A. Setup — Drive (output), repo, deps, inputs

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT_ROOT = '/content/drive/MyDrive/dismantle_final_push'
import os; os.makedirs(OUT_ROOT, exist_ok=True)
REPO_URL='https://github.com/joshuahickscorp/dismantle.git'; BRANCH='codex/maximal-spec-colab'
if not os.path.isdir('/content/dismantle'):
    !git clone --depth 1 --branch $BRANCH $REPO_URL /content/dismantle
else:
    !cd /content/dismantle && git fetch --depth 1 origin $BRANCH && git checkout $BRANCH && git reset --hard origin/$BRANCH
!pip -q install pyarrow safetensors gguf packaging huggingface_hub
import sys; sys.path.insert(0,'/content/dismantle/colab')
print('ready; output ->', OUT_ROOT)


### A2. Fetch corpora (release) + rebuild frozen (HF GGUF, verified)

In [ ]:
REPO_SLUG='joshuahickscorp/dismantle'; RELEASE_TAG='headbank-corpus-v1'
MODELS={
    "q05b": {
        "capture_layer": 23,
        "label": "Qwen2.5-0.5B-Instruct",
        "hf_repo": "Qwen/Qwen2.5-0.5B-Instruct-GGUF",
        "hf_file": "qwen2.5-0.5b-instruct-q4_k_m.gguf"
    },
    "q1p5b": {
        "capture_layer": 27,
        "label": "Qwen2.5-1.5B-Instruct",
        "hf_repo": "Qwen/Qwen2.5-1.5B-Instruct-GGUF",
        "hf_file": "qwen2.5-1.5b-instruct-q4_k_m.gguf"
    },
    "q3b": {
        "capture_layer": 35,
        "label": "Qwen2.5-3B-Instruct",
        "hf_repo": "Qwen/Qwen2.5-3B-Instruct-GGUF",
        "hf_file": "qwen2.5-3b-instruct-q4_k_m.gguf"
    },
    "q7b": {
        "capture_layer": 27,
        "label": "Qwen2.5-7B-Instruct",
        "hf_repo": "bartowski/Qwen2.5-7B-Instruct-GGUF",
        "hf_file": "Qwen2.5-7B-Instruct-Q4_K_M.gguf"
    }
}
import os, json, tarfile, urllib.request, subprocess, numpy as np
from huggingface_hub import hf_hub_download
DATA_ROOT='/content/headbank_inputs'
BASE=f'https://github.com/{REPO_SLUG}/releases/download/{RELEASE_TAG}'
BUILDER='/content/dismantle/tools/orchestrator/build_frozen_gguf.py'
os.makedirs(DATA_ROOT, exist_ok=True)
fp_path=os.path.join(DATA_ROOT,'frozen_fingerprints.json')
urllib.request.urlretrieve(f'{BASE}/frozen_fingerprints.json', fp_path)
FP=json.load(open(fp_path))
READY=[]
for slug,cfg in MODELS.items():
    dst=os.path.join(DATA_ROOT,slug); shards=os.path.join(dst,'corpus_shards')
    os.makedirs(shards, exist_ok=True); frozen=os.path.join(dst,'frozen_gguf.npz')
    if not any(f.endswith('.parquet') for f in os.listdir(shards)):
        tarp=os.path.join(dst,'corpus.tar')
        urllib.request.urlretrieve(f'{BASE}/{slug}_corpus.tar', tarp)
        with tarfile.open(tarp) as t: t.extractall(shards)
        os.remove(tarp)
    if not os.path.isfile(frozen):
        gguf=hf_hub_download(repo_id=cfg['hf_repo'], filename=cfg['hf_file'])
        r=subprocess.run(['python',BUILDER,'--gguf',gguf,'--out',frozen],capture_output=True,text=True)
        if r.returncode!=0: print(slug,'build_frozen failed:',r.stderr[-300:]); continue
    z=np.load(frozen); on=z['output_norm'].astype(np.float32)
    ref=np.array(FP[slug]['output_norm'],dtype=np.float32)
    ok=on.shape==ref.shape and float(np.abs(on-ref).max())<1e-3
    n=len([f for f in os.listdir(shards) if f.endswith('.parquet')])
    print(f'{slug}: shards={n} frozen_verified={ok}')
    if ok and n>0: READY.append(slug)
print('READY:',READY)


## B. HEAD MAXIMIZATION SWEEP  *(the core heavy compute)*
For each model, train a grid of head configs (chained-hidden rollout) and keep the one with the highest **accepted-prefix** (the runtime-predictive spec multiplier). Ordered cheap→expensive so a time-boxed run still produces a winner.

In [ ]:
SWEEP=[[1, 4.0, "1,2,3,4", 12], [2, 4.0, "1,2,3,4", 12], [1, 4.0, "1,2,3,4,5,6", 14], [2, 4.0, "1,2,3,4,5,6", 14]]
TRAINER='/content/dismantle/colab/eagle5_train_pytorch.py'
EVAL='/content/dismantle/colab/eagle5_tau_eval_pytorch.py'
import subprocess, os, json
BEST={}; LEADER=[]
for slug in READY:
    cfg=MODELS[slug]; shards=os.path.join(DATA_ROOT,slug,'corpus_shards')
    frozen=os.path.join(DATA_ROOT,slug,'frozen_gguf.npz')
    best_prefix=-1.0; best_dir=None
    for (nb,ffm,rt,ep) in SWEEP:
        tag=f'b{nb}_ff{ffm}_rt{rt.replace(",","-")}_e{ep}'
        ckpt=os.path.join(OUT_ROOT,'sweep',slug,tag); os.makedirs(ckpt,exist_ok=True)
        tr=['python',TRAINER,'--corpus-dir',shards,'--frozen',frozen,'--ckpt-dir',ckpt,
            '--device','cuda','--target-mode','corpus','--capture-layer',str(cfg['capture_layer']),
            '--epochs',str(ep),'--batch-size','24','--seq-len','16','--lr','1e-3',
            '--num-blocks',str(nb),'--head-ff-mult',str(ffm),
            '--rollout-loss-weight','1.0','--rollout-depth','8',
            '--rollout-depth-targets',rt,'--rollout-draft-prob','0.75',
            '--rollout-chain-hidden','--save-safetensors']
        r=subprocess.run(tr,capture_output=True,text=True)
        if r.returncode!=0: print(f'{slug}/{tag} TRAIN FAIL:',r.stderr[-300:]); continue
        ev=['python',EVAL,'--ckpt',os.path.join(ckpt,'latest.npz'),'--frozen',frozen,
            '--corpus',shards,'--out',os.path.join(ckpt,'tau.json'),'--device','cuda',
            '--target-mode','corpus','--chain-hidden','--depth',str(min(8,max(4,int(rt.split(",")[-1])))),
            '--num-blocks',str(nb),'--head-ff-mult',str(ffm)]
        re=subprocess.run(ev,capture_output=True,text=True)
        prefix=-1.0
        tj=os.path.join(ckpt,'tau.json')
        if re.returncode==0 and os.path.isfile(tj):
            d=json.load(open(tj)); prefix=d.get('tau',-1.0)
        LEADER.append({'slug':slug,'config':tag,'accepted_prefix':prefix,
                       'depth1':json.load(open(tj)).get('depth1_accept_rate') if os.path.isfile(tj) else None})
        print(f'{slug:6s} {tag:28s} accepted_prefix={prefix:.2f} -> ~{1+prefix:.2f}x')
        if prefix>best_prefix: best_prefix=prefix; best_dir=ckpt
    if best_dir: BEST[slug]={'dir':best_dir,'accepted_prefix':best_prefix}
    print(f'=== {slug} BEST: {best_dir} accepted_prefix={best_prefix:.2f} ===')
print('\nBEST per model:', {k:round(v["accepted_prefix"],2) for k,v in BEST.items()})


## C. SPARSITY PREDICTOR  *(experimental — default OFF; audit first)*
The DENOMINATOR lever: predict which FFN weight blocks are active for a token so the runtime skips the rest (read fewer bytes/token → direct tps gain that multiplies with spec). **Requires a local FFN-activation capture** (`dismantle ... --capture-ffn <path>`, NOT yet built — see the after-steps). Set `RUN_SPARSITY=True` only once that capture exists and is uploaded as `<slug>/ffn_act_shards/`. Trainer:
`colab/sparsity_predictor_train.py` (committed in the repo).

In [ ]:
RUN_SPARSITY=False  # AUDIT: enable only when local FFN-activation capture exists
if RUN_SPARSITY:
    SP='/content/dismantle/colab/sparsity_predictor_train.py'
    import subprocess,os
    for slug in READY:
        ffn=os.path.join(DATA_ROOT,slug,'ffn_act_shards')
        frozen=os.path.join(DATA_ROOT,slug,'frozen_gguf.npz')
        if not os.path.isdir(ffn): print(slug,'no ffn capture; skip'); continue
        out=os.path.join(OUT_ROOT,'sparsity',slug); os.makedirs(out,exist_ok=True)
        r=subprocess.run(['python',SP,'--ffn-dir',ffn,'--frozen',frozen,'--out-dir',out,
                           '--device','cuda','--epochs','6','--block-size','256'],
                          capture_output=True,text=True)
        print(slug, 'sparsity:', r.stdout[-400:] if r.returncode==0 else r.stderr[-400:])
else:
    print('sparsity track OFF (default). See after-steps for the local FFN capture.')


## D. Q2/Q3 DISTILLED DRAFT  *(experimental — default OFF; audit first)*
A cheaper standalone draft (small + low-bit) so the runtime can afford deeper/wider TREE drafts. The chained head already serves as the draft; this is a stretch lever. Default OFF; documented for the audit. A distillation harness would train a tiny model to mimic the target's next-token distribution on the corpus, then quantize to Q3_K/Q2_K.

In [ ]:
RUN_Q2_DRAFT=False  # AUDIT: design + validate before spending compute
print('Q2 draft track OFF (default). Stretch lever; the chained head is the primary draft.')


## E. Emit best heads + leaderboard + manifest

In [ ]:
import json,os,shutil,hashlib
def sha(p):
    h=hashlib.sha256();
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1<<20),b''): h.update(b)
    return h.hexdigest()
FINAL=os.path.join(OUT_ROOT,'best_heads'); os.makedirs(FINAL,exist_ok=True)
entries=[]
for slug,info in BEST.items():
    src=os.path.join(info['dir'],'head_final.safetensors')
    if not os.path.isfile(src): continue
    dstd=os.path.join(FINAL,slug); os.makedirs(dstd,exist_ok=True)
    dst=os.path.join(dstd,'head_final.safetensors'); shutil.copy2(src,dst)
    entries.append({'slug':slug,'label':MODELS[slug]['label'],
                    'capture_layer':MODELS[slug]['capture_layer'],
                    'accepted_prefix':info['accepted_prefix'],
                    'projected_speedup':round(1+info['accepted_prefix'],2),
                    'head_path':dst,'head_sha256':sha(dst)})
json.dump({'schema':'dismantle-final-push-v1','entries':entries,'leaderboard':LEADER},
          open(os.path.join(FINAL,'manifest.json'),'w'),indent=2)
print('=== FINAL HEADBANK ===')
for e in entries: print(f"{e['slug']:6s} accepted_prefix={e['accepted_prefix']:.2f} -> ~{e['projected_speedup']}x  {e['head_path']}")
print('\nFull leaderboard + heads in', FINAL)
